# Evaluation Metrics

This notebook covers all three module evaluations and suggests additional metrics worth adding to the report.

## What's here

| Section | What it does | Status |
|---------|-------------|--------|
| Module A | Aspect classification: sample generation + metric computation | Needs manual labels |
| Module B | Regression RMSE/MAE/R², SHAP stability across 9 quarters | Auto-computed |
| Module C | Agent: groundedness, resolution rate, GPT-4o relevance judge | Runs live agent |

## Additional metrics suggested
- **Module A**: Cohen's kappa (inter-annotator agreement), per-aspect confusion matrix, keyword coverage rate
- **Module B**: Kendall's tau between quarterly SHAP rankings, per-hotel SHAP variance, feature importance correlation between linear and XGB
- **Module C**: Mean retrieval cosine similarity, fallback rate, citation coverage rate (% responses with ≥1 citation), average tokens per response

In [ ]:
import os, sys, json, random
import numpy as np
import pandas as pd
from scipy import stats

NOTEBOOK_DIR = os.path.abspath('')
ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC  = os.path.join(ROOT, 'src')
for p in (ROOT, SRC):
    if p not in sys.path:
        sys.path.insert(0, p)

from paths import (SENTENCES, ASPECT_SENTENCES, SHAP_SUMMARY,
                   REVIEW_FEATURES, EVALUATION_REPORT)
from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, '.env'))

print('Imports OK')

---
# Module A: Aspect Classification Evaluation

Sentiment is assigned deterministically from source field, so evaluation focuses on **aspect assignment quality**: did the keyword matcher pick the right aspect for each sentence?

### Step 1 — Generate a stratified validation sample
Run the cell below to produce `outputs/validation_sample.csv` with 30 sentences per aspect (180 total), stratified by aspect. Then open the file, add a `true_aspect` column, and label each row manually before running Step 2.

In [ ]:
SAMPLE_PER_ASPECT = 30
VALIDATION_PATH   = os.path.join(ROOT, 'outputs', 'validation_sample.csv')
ASPECTS = ['Room', 'Staff', 'Location', 'Food', 'Cleanliness', 'Noise']

df_asp = pd.read_csv(ASPECT_SENTENCES, low_memory=False)

random.seed(42)
samples = []
for asp in ASPECTS:
    subset = df_asp[df_asp['aspect'] == asp].sample(
        n=min(SAMPLE_PER_ASPECT, len(df_asp[df_asp['aspect'] == asp])),
        random_state=42
    )[['review_id', 'hotel_name', 'sentence', 'aspect', 'sentiment', 'reviewer_segment']]
    samples.append(subset)

val_df = pd.concat(samples, ignore_index=True).sample(frac=1, random_state=42)
val_df['true_aspect'] = ''   # to be filled in manually
val_df.to_csv(VALIDATION_PATH, index=False)

print(f'Saved {len(val_df)} rows to {VALIDATION_PATH}')
print('Fill the `true_aspect` column with the correct aspect label, then run Step 2.')
val_df[['sentence', 'aspect', 'true_aspect']].head(10)

In [ ]:
# ── Step 2: Compute classification metrics from labelled validation set ────────
# Run this cell AFTER filling in the `true_aspect` column in validation_sample.csv

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, cohen_kappa_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

if not os.path.isfile(VALIDATION_PATH):
    print('Run Step 1 first to generate the validation sample.')
else:
    val = pd.read_csv(VALIDATION_PATH)
    labelled = val[val['true_aspect'].str.strip() != '']

    if len(labelled) == 0:
        print('No labels found in validation_sample.csv. Fill the `true_aspect` column first.')
    else:
        y_pred = labelled['aspect'].str.strip()
        y_true = labelled['true_aspect'].str.strip()

        acc   = accuracy_score(y_true, y_pred)
        kappa = cohen_kappa_score(y_true, y_pred)
        p, r, f, support = precision_recall_fscore_support(
            y_true, y_pred, labels=ASPECTS, average=None, zero_division=0
        )
        p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0
        )

        print(f'Overall accuracy : {acc:.4f}')
        print(f"Cohen's kappa    : {kappa:.4f}")
        print(f'Macro precision  : {p_macro:.4f}')
        print(f'Macro recall     : {r_macro:.4f}')
        print(f'Macro F1         : {f_macro:.4f}')
        print()
        print(classification_report(y_true, y_pred, labels=ASPECTS, zero_division=0))

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred, labels=ASPECTS)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=ASPECTS,
                    yticklabels=ASPECTS, cmap='Blues', ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('Aspect classification confusion matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(ROOT, 'docs', 'figures', 'confusion_matrix.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# ── Additional Module A metric: keyword coverage rate ────────────────────────
# What fraction of all sentences matched an aspect at all?

df_sent = pd.read_csv(SENTENCES, low_memory=False)
total_sentences  = len(df_sent)
matched_sentences = len(df_asp)

print(f'Total sentences:   {total_sentences:,}')
print(f'Matched sentences: {matched_sentences:,}')
print(f'Coverage rate:     {matched_sentences/total_sentences*100:.1f}%')
print(f'Unmatched:         {total_sentences - matched_sentences:,} sentences had no keyword match')
print()
print('Per-aspect coverage:')
for asp in ASPECTS:
    n = (df_asp['aspect'] == asp).sum()
    print(f'  {asp:12s}: {n:,} sentences ({n/total_sentences*100:.1f}% of all)')

---
# Module B: Rating Impact Model Evaluation

In [ ]:
# ── Load evaluation report ───────────────────────────────────────────────────
with open(EVALUATION_REPORT) as f:
    report = json.load(f)

print('=== Model Performance (80/20 split, random_state=42) ===')
for model, metrics in report.items():
    if model == 'shap_stability':
        continue
    print(f'\n{model.replace("_", " ").title()}')
    print(f'  RMSE : {metrics["rmse"]:.4f}')
    print(f'  MAE  : {metrics["mae"]:.4f}')
    print(f'  R²   : {metrics["r2"]:.4f}')

print('\n=== SHAP Rank Stability ===')
stab = report['shap_stability']
print(f'Top-3 consistent across all slices: {stab["top_3_consistent_across_slices"]}')
print(f'Number of time slices: {len(stab["slice_rankings"])}')
print('\nQuarterly top-3 aspects:')
for s in stab['slice_rankings']:
    print(f'  {s["period"]}: {s["top_aspects"]}')

In [ ]:
# ── Additional Module B metric: Kendall's tau between quarterly rankings ──────
# Measures how similar the full 6-aspect ranking is across time slices (not just top-3)

import pickle
import shap as shap_lib
from paths import XGB_MODEL

with open(XGB_MODEL, 'rb') as f:
    xgb_model = pickle.load(f)

df_feat = pd.read_csv(REVIEW_FEATURES, low_memory=False)
df_sent_dates = pd.read_csv(SENTENCES, usecols=['review_id', 'review_date'],
                             low_memory=False).drop_duplicates(subset='review_id')
df_dated = df_feat.merge(df_sent_dates, on='review_id', how='left').dropna(subset=['review_date'])
df_dated['quarter'] = pd.to_datetime(df_dated['review_date'], errors='coerce').dt.to_period('Q').astype(str)

ASPECT_COLS = ['cleanliness', 'staff', 'location', 'noise', 'food', 'room']
explainer = shap_lib.TreeExplainer(xgb_model)

quarterly_ranks = {}
for period, group in sorted(df_dated.groupby('quarter')):
    X_slice = group[ASPECT_COLS].values.astype(float)
    sv = explainer.shap_values(X_slice)
    mean_abs = np.abs(sv).mean(axis=0)
    rank = pd.Series(mean_abs, index=ASPECT_COLS).rank(ascending=False).astype(int)
    quarterly_ranks[period] = rank

# Compute pairwise Kendall's tau
periods = list(quarterly_ranks.keys())
taus = []
for i in range(len(periods)):
    for j in range(i+1, len(periods)):
        tau, _ = stats.kendalltau(quarterly_ranks[periods[i]], quarterly_ranks[periods[j]])
        taus.append(tau)

print(f'Mean Kendall\'s tau across {len(taus)} quarterly pairs: {np.mean(taus):.4f}')
print(f'Min tau: {min(taus):.4f}  Max tau: {max(taus):.4f}')
print(f'\nInterpretation: tau=1.0 means identical ranking, 0.0 means random')
print('Quarterly SHAP ranks (1=highest impact):')
rank_df = pd.DataFrame(quarterly_ranks).T
rank_df.columns = [c.capitalize() for c in rank_df.columns]
print(rank_df.to_string())

In [ ]:
# ── Additional: linear vs XGB SHAP correlation ───────────────────────────────
import pickle
from paths import LINEAR_MODEL

with open(LINEAR_MODEL, 'rb') as f:
    lin_model = pickle.load(f)

# Linear model coefficients as a proxy for linear SHAP importance
lin_coefs = pd.Series(np.abs(lin_model.coef_), index=ASPECT_COLS)

with open(SHAP_SUMMARY) as f:
    shap_data = json.load(f)
global_entry = next(e for e in shap_data if e['hotel_name'] == '__global__')
xgb_shap_abs = pd.Series(
    {k: abs(v) for k, v in global_entry['aspect_impacts'].items()}
)

combined = pd.DataFrame({
    'Linear |coef|': lin_coefs,
    'XGB |SHAP|':    xgb_shap_abs
})
tau, pval = stats.kendalltau(lin_coefs.rank(), xgb_shap_abs.rank())
print('Aspect importance: Linear coefficients vs XGBoost SHAP')
print(combined.to_string())
print(f"\nKendall's tau (rank agreement): {tau:.4f}  (p={pval:.4f})")

---
# Module C: Conversational Agent Evaluation

We evaluate on a curated set of 20 queries covering all four query types: evidence, prioritisation, mismatch, and segment.

**Metrics:**
| Metric | Definition | Method |
|--------|-----------|--------|
| Groundedness | Fraction of responses with ≥1 citation | Automatic (check `citations` field) |
| Resolution rate | Fraction answered without low-confidence fallback | Automatic (check `low_confidence` field) |
| Mean retrieval similarity | Average cosine similarity of retrieved chunks | Automatic (from `citations`) |
| Relevance | 1–5 score from GPT-4o judge | GPT-4o API call |
| Context accuracy | GPT-4o judge: does response correctly use retrieved evidence? | GPT-4o API call |

> **Note:** ChromaDB must be populated (run `python scripts/run_pipeline.py --only 5`) before running agent cells.

In [ ]:
# ── Evaluation query set ──────────────────────────────────────────────────────
EVAL_QUERIES = [
    # Evidence queries — what do guests say?
    {'query': 'What do guests complain about regarding cleanliness?',
     'hotel': '__global__', 'type': 'evidence'},
    {'query': 'What aspects of the staff do guests praise most?',
     'hotel': '__global__', 'type': 'evidence'},
    {'query': 'What noise problems do guests mention?',
     'hotel': '__global__', 'type': 'evidence'},
    {'query': 'What do guests say about the location?',
     'hotel': '__global__', 'type': 'evidence'},
    {'query': 'What complaints do guests have about the rooms?',
     'hotel': '__global__', 'type': 'evidence'},

    # Prioritisation queries — which aspect to fix first?
    {'query': 'Which aspect should this hotel prioritise improving?',
     'hotel': '41', 'type': 'prioritisation'},
    {'query': 'What is dragging down ratings the most globally?',
     'hotel': '__global__', 'type': 'prioritisation'},
    {'query': 'Which aspect has the biggest positive impact on guest scores?',
     'hotel': '__global__', 'type': 'prioritisation'},
    {'query': 'What should Hotel Arena focus on to improve its rating?',
     'hotel': 'Hotel Arena', 'type': 'prioritisation'},

    # Mismatch queries — sentiment vs score contradiction
    {'query': 'Are there guests who gave positive reviews but low scores?',
     'hotel': '__global__', 'type': 'mismatch'},
    {'query': 'Which aspects show the biggest mismatch between text and rating?',
     'hotel': '__global__', 'type': 'mismatch'},

    # Segment queries — reviewer type specific
    {'query': 'What do business travellers complain about most?',
     'hotel': '__global__', 'type': 'segment'},
    {'query': 'What do families say about the rooms?',
     'hotel': '__global__', 'type': 'segment'},
    {'query': 'What aspects do couples rate most positively?',
     'hotel': '__global__', 'type': 'segment'},
    {'query': 'What do solo travellers complain about regarding noise?',
     'hotel': '__global__', 'type': 'segment'},

    # Edge cases — should trigger fallback or graceful handling
    {'query': 'What do guests say about the infinity pool and spa facilities?',
     'hotel': '__global__', 'type': 'evidence'},   # out-of-vocabulary topic
    {'query': 'Tell me about Hotel Xyzabc Nonexistent 1234',
     'hotel': 'Hotel Xyzabc Nonexistent 1234', 'type': 'evidence'},  # unresolvable hotel
    {'query': 'What is the price range for rooms?',
     'hotel': '__global__', 'type': 'evidence'},   # topic not in dataset
    {'query': 'How is the cleanliness at 11 Cadogan Gardens?',
     'hotel': '11 Cadogan Gardens', 'type': 'evidence'},
    {'query': 'Which aspect matters most for business travellers at The Savoy?',
     'hotel': 'The Savoy', 'type': 'segment'},
]

print(f'{len(EVAL_QUERIES)} evaluation queries across 4 types:')
for t in ['evidence', 'prioritisation', 'mismatch', 'segment']:
    n = sum(1 for q in EVAL_QUERIES if q['type'] == t)
    print(f'  {t:15s}: {n}')

In [ ]:
# ── Run queries through agent ─────────────────────────────────────────────────
# Requires chromadb/ to be populated and OPENAI_API_KEY in .env

from agent.graph import run_query

results = []
for i, q in enumerate(EVAL_QUERIES):
    print(f'[{i+1}/{len(EVAL_QUERIES)}] {q["type"]:15s} | {q["query"][:60]}...')
    try:
        result = run_query(
            query=q['query'],
            hotel_name=q['hotel'],
            thread_id=f'eval_{i}',  # each query gets its own thread (no memory bleed)
        )
        results.append({
            'query_type':  q['type'],
            'query':       q['query'],
            'hotel':       q['hotel'],
            'response':    result.get('response', ''),
            'citations':   result.get('citations', []),
            'low_confidence': result.get('low_confidence', False),
            'hotel_unresolved': result.get('hotel_unresolved', False),
        })
    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'query_type': q['type'], 'query': q['query'],
                        'hotel': q['hotel'], 'response': '', 'citations': [],
                        'low_confidence': True, 'hotel_unresolved': False, 'error': str(e)})

print(f'\nDone. {len(results)} results.')

In [ ]:
# ── Automatic metrics (no human/LLM needed) ───────────────────────────────────

df_res = pd.DataFrame(results)

# 1. Resolution rate — fraction answered without low-confidence fallback
resolution_rate = (~df_res['low_confidence']).mean()

# 2. Citation coverage — fraction of non-fallback responses with ≥1 citation
confident = df_res[~df_res['low_confidence']]
citation_coverage = confident['citations'].apply(lambda c: len(c) > 0).mean() if len(confident) > 0 else 0

# 3. Mean retrieval similarity (from citations)
all_sims = [
    c.get('similarity_score', 0)
    for row in results
    for c in row['citations']
]
mean_sim = np.mean(all_sims) if all_sims else 0

# 4. Per-type resolution rate
per_type = df_res.groupby('query_type').apply(
    lambda g: (~g['low_confidence']).mean()
)

# 5. Hotel unresolved rate
unresolved_rate = df_res['hotel_unresolved'].mean()

print('=== Automatic Metrics ===')
print(f'Resolution rate (overall):  {resolution_rate:.2%}')
print(f'Citation coverage:          {citation_coverage:.2%}  (of resolved responses)')
print(f'Mean retrieval similarity:  {mean_sim:.4f}')
print(f'Hotel unresolved rate:      {unresolved_rate:.2%}')
print()
print('Resolution rate by query type:')
print(per_type.to_string())

In [ ]:
# ── GPT-4o judge: Relevance and Context Accuracy ─────────────────────────────
# Judge prompt adapted from G-Eval (Liu et al., 2023)

from openai import OpenAI
import re

client = OpenAI()

JUDGE_PROMPT = """You are an evaluation judge for a hotel analytics AI assistant.
Score the response on two dimensions. Return ONLY a JSON object with keys
"relevance" and "context_accuracy", each an integer 1-5.

Relevance (1-5): Does the response directly answer the user's question?
  5=perfectly on-topic  4=mostly  3=partial  2=tangential  1=off-topic

Context accuracy (1-5): Does the response correctly use the retrieved evidence?
  5=all claims grounded and accurate  4=mostly  3=some unsupported claims
  2=mostly ungrounded  1=ignores evidence or hallucinates

Question: {query}
Response: {response}
Retrieved sources used: {n_citations} citations"""

def judge(query, response, n_citations):
    if not response.strip():
        return {'relevance': 1, 'context_accuracy': 1}
    prompt = JUDGE_PROMPT.format(
        query=query, response=response[:800], n_citations=n_citations
    )
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=60,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except Exception:
        nums = re.findall(r'[1-5]', raw)
        return {'relevance': int(nums[0]) if len(nums) > 0 else 3,
                'context_accuracy': int(nums[1]) if len(nums) > 1 else 3}

print('Running GPT-4o judge on all responses...')
for i, row in enumerate(results):
    scores = judge(row['query'], row['response'], len(row['citations']))
    results[i]['relevance']        = scores.get('relevance', 3)
    results[i]['context_accuracy'] = scores.get('context_accuracy', 3)
    print(f"  [{i+1}/{len(results)}] R={results[i]['relevance']} CA={results[i]['context_accuracy']}")

print('Done.')

In [ ]:
# ── Summary table — all Module C metrics ─────────────────────────────────────

df_res = pd.DataFrame(results)

metrics = {
    'Groundedness (citation coverage)': f"{df_res[~df_res['low_confidence']]['citations'].apply(lambda c: len(c) > 0).mean():.2%}",
    'Relevance (GPT-4o judge, 1–5)':   f"{df_res['relevance'].mean():.2f} ± {df_res['relevance'].std():.2f}",
    'Context accuracy (GPT-4o, 1–5)':  f"{df_res['context_accuracy'].mean():.2f} ± {df_res['context_accuracy'].std():.2f}",
    'Resolution rate':                  f"{(~df_res['low_confidence']).mean():.2%}",
    'Mean retrieval similarity':        f"{np.mean([c.get('similarity_score',0) for r in results for c in r['citations']]):.4f}",
}

print('=== Module C Evaluation Summary ===')
for k, v in metrics.items():
    print(f'  {k:45s}: {v}')

print('\nPer query type:')
for t in ['evidence', 'prioritisation', 'mismatch', 'segment']:
    sub = df_res[df_res['query_type'] == t]
    print(f"  {t:15s}: resolution={( ~sub['low_confidence']).mean():.0%}  "
          f"relevance={sub['relevance'].mean():.1f}  "
          f"context_acc={sub['context_accuracy'].mean():.1f}")

# Save results
df_res.to_csv(os.path.join(ROOT, 'outputs', 'agent_eval_results.csv'), index=False)
print('\nSaved to outputs/agent_eval_results.csv')

In [ ]:
# ── Visualise Module C results ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib

fig, axes = plt.subplots(1, 3, figsize=(9, 3))

# Relevance distribution
axes[0].hist(df_res['relevance'], bins=[0.5,1.5,2.5,3.5,4.5,5.5],
             color='#4C72B0', edgecolor='white', rwidth=0.8)
axes[0].set_xticks([1,2,3,4,5])
axes[0].set_xlabel('Score (1–5)')
axes[0].set_title(f'Relevance (mean={df_res["relevance"].mean():.2f})')

# Context accuracy distribution
axes[1].hist(df_res['context_accuracy'], bins=[0.5,1.5,2.5,3.5,4.5,5.5],
             color='#55A868', edgecolor='white', rwidth=0.8)
axes[1].set_xticks([1,2,3,4,5])
axes[1].set_xlabel('Score (1–5)')
axes[1].set_title(f'Context accuracy (mean={df_res["context_accuracy"].mean():.2f})')

# Resolution rate by type
types = ['evidence', 'prioritisation', 'mismatch', 'segment']
rates = [(~df_res[df_res['query_type']==t]['low_confidence']).mean() for t in types]
axes[2].bar(types, rates, color=['#4C72B0','#55A868','#C44E52','#8172B2'], edgecolor='white')
axes[2].set_ylim(0, 1.1)
axes[2].set_ylabel('Resolution rate')
axes[2].set_title('Resolution rate by query type')
axes[2].tick_params(axis='x', labelsize=7.5)
for bar, rate in zip(axes[2].patches, rates):
    axes[2].text(bar.get_x() + bar.get_width()/2, rate + 0.02,
                 f'{rate:.0%}', ha='center', fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'docs', 'figures', 'agent_eval.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/agent_eval.png')

---
## Report table fill-in

Copy these numbers into `docs/report_draft.md`:

### Table 4 — Module A (fill after manual labelling)
```
Run the validation sample cells above, fill true_aspect, then copy
the per-aspect Accuracy/Precision/Recall/F1 from the classification report.
```

### Table 5 — Module B (already filled)
```
Linear Regression: RMSE=1.4125  MAE=1.0857  R²=0.2464
XGBoost:           RMSE=1.4014  MAE=1.0756  R²=0.2582
SHAP top-3 consistent across all 9 quarters: True
```

### Table 7 — Module C (fill after running agent cells)
```
Groundedness:    see citation_coverage value above
Relevance:       see GPT-4o judge mean above
Context accuracy: see GPT-4o judge mean above
Resolution rate: see resolution_rate above
```